In [6]:
import pandas as pd
import json
import re

# ==========================================
# 1. 파일 경로 설정 (사용자 환경에 맞게 수정)
# ==========================================
file_ranked_csv = 'result_ranked.csv'   # 순위가 매겨진 CSV 파일
file_raw_json = '../SSU_Datathon2025_공학분야_62199.json'           # 원본 대용량 JSON 파일 (이름 확인 필요)
output_json = 'top_30_percent_papers.json' # 결과로 저장될 파일명

In [7]:
# ==========================================
# 2. 순위 파일(CSV) 파싱 함수
# ==========================================
def parse_ranked_journals(rank_str):
    """
    '1. 학술지명 (점수)\n2. 학술지명 (점수)...' 형태의 문자열에서
    순서대로 [학술지명1, 학술지명2, ...] 리스트를 추출합니다.
    """
    if pd.isna(rank_str) or rank_str == "":
        return []
    
    journal_list = []
    # 줄바꿈(\n)으로 분리하여 각 라인 처리
    for line in rank_str.split('\n'):
        # 정규표현식으로 "숫자. " 과 맨 뒤의 " (점수)" 제거
        # 예: "1. 한국컴퓨터학회 (1.5)" -> "한국컴퓨터학회"
        # 패턴: 숫자. (그룹) (숫자)
        match = re.match(r'\d+\.\s(.+)\s\(', line)
        if match:
            journal_name = match.group(1).strip()
            journal_list.append(journal_name)
        else:
            # 매칭 실패 시 (혹시 포맷이 다를 경우) 단순 분할 시도
            try:
                # 맨 앞 "1. " 제거 시도
                temp = line.split('. ', 1)[1]
                # 맨 뒤 " (" 제거 시도
                name = temp.rsplit(' (', 1)[0]
                journal_list.append(name.strip())
            except:
                continue
                
    return journal_list

In [9]:
# ==========================================
# 3. 메인 로직 실행
# ==========================================
try:
    print("Step 1: 순위 파일(CSV)을 로드하고 파싱합니다...")
    df_rank = pd.read_csv(file_ranked_csv)
    df_rank.columns = df_rank.columns.str.strip()
    
    # 학회_순위(IF) 컬럼을 파싱하여 리스트로 변환
    df_rank['Sorted_Journals'] = df_rank['학회_순위(IF)'].apply(parse_ranked_journals)

    print("Step 2: 원본 JSON 데이터를 로드합니다 (시간이 걸릴 수 있습니다)...")
    # 대용량 JSON 로드 (형식에 따라 pd.read_json 또는 json.load 사용)
    with open(file_raw_json, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    # 딕셔너리 안에 리스트가 있는 경우 처리 (이전 대화 맥락 반영)
    if isinstance(data, dict):
        for key, val in data.items():
            if isinstance(val, list):
                df_raw = pd.DataFrame(val)
                break
    else:
        df_raw = pd.DataFrame(data)

    # 전처리: 연도 컬럼 생성
    df_raw['Year'] = df_raw['PBSH'].astype(str).str[:4]
    
    # 전처리: 컬럼 공백 제거 (안전을 위해)
    df_raw['NODE_CLSS_02'] = df_raw['NODE_CLSS_02'].str.strip()
    df_raw['PLCT_NM'] = df_raw['PLCT_NM'].str.strip() # 학술지명

    print("Step 3: 연도x중분류x학술지별 논문 수를 계산합니다...")
    # 그룹별 논문 수 카운트
    # 예: 2021 | 컴퓨터공학 | 한국컴퓨터학회 | 100편
    journal_counts = df_raw.groupby(['Year', 'NODE_CLSS_02', 'PLCT_NM']).size().reset_index(name='Count')

    # 빠른 조회를 위해 MultiIndex로 변환
    journal_counts.set_index(['Year', 'NODE_CLSS_02', 'PLCT_NM'], inplace=True)

    print("Step 4: 상위 30% (Min 50) 추출 로직을 수행합니다...")
    
    selected_indices = [] # 선택된 논문의 인덱스를 담을 리스트

    for idx, row in df_rank.iterrows():
        try:
            year = str(int(float(row['Year'])))
        except:
            continue
            
        category = row['NODE_CLSS_02']
        sorted_journals = row['Sorted_Journals']
        total_papers = row['논문수']
        
        # 목표 개수 설정
        target_count = max(int(total_papers * 0.30), 50)
        if total_papers < 50: target_count = total_papers
        
        current_count = 0
        
        # 원본 데이터에서 해당 그룹(연도/중분류) 데이터만 먼저 필터링 (속도 최적화)
        group_df = df_raw[(df_raw['Year'] == year) & (df_raw['NODE_CLSS_02'] == category)]
        
        for journal in sorted_journals:
            # 해당 학술지의 논문들 찾기
            journal_papers = group_df[group_df['PLCT_NM'] == journal]
            
            if not journal_papers.empty:
                # 남은 필요한 개수 계산
                needed = target_count - current_count
                
                if len(journal_papers) <= needed:
                    # 필요한 것보다 적으면 통째로 다 추가
                    selected_indices.extend(journal_papers.index.tolist())
                    current_count += len(journal_papers)
                else:
                    # 필요한 것보다 많으면 딱 필요한 만큼만 잘라서 추가 (랜덤 or 앞순서)
                    # 여기서는 앞에서부터 자름
                    sliced_indices = journal_papers.head(needed).index.tolist()
                    selected_indices.extend(sliced_indices)
                    current_count += needed
            
            # 목표 달성하면 중단
            if current_count >= target_count:
                break
                
    # 최종 필터링
    filtered_df = df_raw.loc[selected_indices].reset_index(drop=True)
    
    # 저장
    output_json = 'top_30_percent_papers_exact.json'
    filtered_df.to_json(output_json, orient='records', force_ascii=False, indent=4)
    print(f"🎉 개수를 딱 맞춰서 저장 완료! ({len(filtered_df)} 건)")

except Exception as e:
    print(f"❌ 오류 발생: {e}")

Step 1: 순위 파일(CSV)을 로드하고 파싱합니다...
Step 2: 원본 JSON 데이터를 로드합니다 (시간이 걸릴 수 있습니다)...
Step 3: 연도x중분류x학술지별 논문 수를 계산합니다...
Step 4: 상위 30% (Min 50) 추출 로직을 수행합니다...
🎉 개수를 딱 맞춰서 저장 완료! (18638 건)


In [11]:
# 확인용

import pandas as pd
import json
import re

# ==========================================
# 1. 파일 경로 설정
# ==========================================
file_result_json = 'top_30_percent_papers_exact.json'  # 생성된 결과 파일
file_ranked_csv = 'result_ranked.csv'            # 기준 순위 파일

# ==========================================
# 2. 순위 파싱 함수 (이전과 동일)
# ==========================================
def parse_ranked_journals(rank_str):
    if pd.isna(rank_str) or rank_str == "":
        return []
    journal_list = []
    for line in rank_str.split('\n'):
        # "1. 학술지명 (점수)" 패턴에서 이름 추출
        match = re.match(r'\d+\.\s(.+)\s\(', line)
        if match:
            journal_list.append(match.group(1).strip())
        else:
            # 패턴 매칭 실패 시 단순 분리 시도
            try:
                temp = line.split('. ', 1)[1]
                name = temp.rsplit(' (', 1)[0]
                journal_list.append(name.strip())
            except:
                continue
    return journal_list

# ==========================================
# 3. 검증 로직 실행
# ==========================================
try:
    print("Step 1: 파일들을 로드합니다...")
    
    # 1. CSV 로드
    df_rank = pd.read_csv(file_ranked_csv)
    df_rank.columns = df_rank.columns.str.strip()
    
    # 2. 결과 JSON 로드
    try:
        df_result = pd.read_json(file_result_json)
    except ValueError:
        # JSON 포맷 이슈 대비
        with open(file_result_json, 'r', encoding='utf-8') as f:
            data = json.load(f)
        df_result = pd.DataFrame(data)

    # JSON 데이터 전처리
    df_result['Year'] = df_result['PBSH'].astype(str).str[:4]
    df_result['NODE_CLSS_02'] = df_result['NODE_CLSS_02'].str.strip()
    df_result['PLCT_NM'] = df_result['PLCT_NM'].str.strip()

    print("\n" + "="*80)
    print(f"{'연도':^6} | {'중분류':^15} | {'전체논문':^8} | {'목표(Min)':^10} | {'실제추출':^10} | {'상태':^6}")
    print("="*80)

    # 그룹별 검증
    for idx, row in df_rank.iterrows():
        try:
            year = str(int(float(row['Year'])))
        except:
            continue
            
        category = row['NODE_CLSS_02']
        total_count = row['논문수']
        rank_str = row['학회_순위(IF)']
        
        # 1. 목표 수량 계산
        if total_count < 50:
            target_min = total_count
        else:
            target_min = max(int(total_count * 0.3), 50)
            
        # 2. 실제 추출된 수량 확인
        extracted_data = df_result[
            (df_result['Year'] == year) & 
            (df_result['NODE_CLSS_02'] == category)
        ]
        actual_count = len(extracted_data)
        
        # 3. 상태 판정 (실제 개수가 목표치보다 크거나 같아야 함)
        # 단, 마지막 학술지의 논문을 다 넣다보면 목표치를 넘길 수 있으므로 '>= target'이면 통과
        status = "✅ OK" if actual_count >= target_min else "⚠️ 부족"
        if total_count == 0: status = "-" # 원본이 없는 경우
        
        # 4. 상위 학술지 포함 여부 검증
        ranked_journals = parse_ranked_journals(rank_str)
        extracted_journals = set(extracted_data['PLCT_NM'].unique())
        
        # 실제로 추출된 학술지들이 순위권 상위에 있는지 확인
        # (상위 1등 학술지가 누락되었는데 10등이 들어갔는지 체크)
        check_rank_msg = ""
        if actual_count > 0 and ranked_journals:
            # 추출된 학술지 중 가장 낮은 순위 찾기
            min_rank_idx = -1
            for j in extracted_journals:
                if j in ranked_journals:
                    idx = ranked_journals.index(j)
                    if idx > min_rank_idx:
                        min_rank_idx = idx
            
            # 이론상 0등부터 min_rank_idx까지는 다 들어와 있어야 함 (데이터가 있다면)
            # 여기서는 간단히 상위 3개 학술지가 포함되었는지 정도만 로그로 확인
            top_3 = ranked_journals[:3]
            missing_top = [j for j in top_3 if j not in extracted_journals]
            # 원본 데이터에 해당 학술지 논문이 아예 없을 수도 있으므로 경고만 표시
            
        print(f"{year:^6} | {category:<15} | {total_count:^8} | {target_min:^10} | {actual_count:^10} | {status}")

    print("="*80)
    print("검증 완료! '실제추출'이 '목표(Min)'보다 크거나 같으면 정상입니다.")
    print("(마지막으로 포함된 학술지의 전체 논문을 가져오기 때문에 실제 개수가 목표보다 조금 많을 수 있습니다.)")

except Exception as e:
    print(f"❌ 검증 중 오류 발생: {e}")

Step 1: 파일들을 로드합니다...

  연도   |       중분류       |   전체논문   |  목표(Min)   |    실제추출    |   상태  
 2021  | 건축공학            |   2402   |    720     |    720     | ✅ OK
 2021  | 공학 일반           |   1128   |    338     |    338     | ✅ OK
 2021  | 기계공학            |   2635   |    790     |    790     | ✅ OK
 2021  | 기타 공학           |   447    |    134     |    134     | ✅ OK
 2021  | 산업공학            |   289    |     86     |     86     | ✅ OK
 2021  | 재료·에너지공학        |   308    |     92     |     92     | ✅ OK
 2021  | 전기전자공학          |   3697   |    1109    |    1109    | ✅ OK
 2021  | 조선해양공학          |   208    |     62     |     62     | ✅ OK
 2021  | 컴퓨터학            |   1327   |    398     |    398     | ✅ OK
 2021  | 화학공학            |   333    |     99     |     99     | ✅ OK
 2022  | 건축공학            |   2333   |    699     |    699     | ✅ OK
 2022  | 공학 일반           |   1090   |    327     |    327     | ✅ OK
 2022  | 기계공학            |   2591   |    777     |    777     | ✅ OK
 2022  | 

In [13]:
# 50개인 논문 확인용

import pandas as pd

# 1. 분석할 파일 경로
input_json_file = 'top_30_percent_papers_exact.json'

try:
    # 데이터 로드
    df = pd.read_json(input_json_file)
    
    # 'Year' 컬럼 확인 및 생성
    if 'Year' not in df.columns:
        df['Year'] = df['PBSH'].astype(str).str[:4]
        
    # 공백 제거
    df['NODE_CLSS_02'] = df['NODE_CLSS_02'].str.strip()

    # 2. [연도 x 중분류] 별 개수 집계
    group_counts = df.groupby(['Year', 'NODE_CLSS_02']).size().reset_index(name='Extracted_Count')

    # 3. 개수가 정확히 50개인 그룹 찾기
    target_groups = group_counts[group_counts['Extracted_Count'] == 50]

    # 4. 결과 화면 출력
    print("\n" + "="*60)
    print(f"📊 추출된 논문이 '정확히 50개'인 그룹 (총 {len(target_groups)}건)")
    print("="*60)
    
    if not target_groups.empty:
        # 보기 좋게 행별로 출력
        print(f"{'연도':^10} | {'중분류':^20} | {'개수':^10}")
        print("-" * 60)
        
        for idx, row in target_groups.iterrows():
            print(f"{row['Year']:^10} | {row['NODE_CLSS_02']:<20} | {row['Extracted_Count']:^10}")
            
        print("-" * 60)
    else:
        print(">> 해당 조건(정확히 50개)에 맞는 그룹이 없습니다.")
        print(">> (대부분 50개를 조금 넘겼을 가능성이 높습니다.)")

except Exception as e:
    print(f"❌ 오류 발생: {e}")


📊 추출된 논문이 '정확히 50개'인 그룹 (총 0건)
>> 해당 조건(정확히 50개)에 맞는 그룹이 없습니다.
>> (대부분 50개를 조금 넘겼을 가능성이 높습니다.)
